# Double Descent

In [ ]:
import sys
sys.path.insert(0, '/Users/aayushjoshi/MyFiles/CS/487/double-descent/hessian-eff-dim')

import math
import torch
import hess
import matplotlib.pyplot as plt
from hess.nets import SimpleNet
import hess.loss_surfaces as loss_surfaces
import numpy as np
import sklearn.datasets as datasets
from sklearn.datasets import make_moons
try:
    from tqdm.auto import tqdm
except Exception:
    from tqdm import tqdm
import hess.utils as utils

## Dataset Preparation

In [ ]:
def twospirals(n_points, noise=.5, random_state=920):
    """
     Returns the two spirals dataset.
    """
    n = np.sqrt(np.random.rand(n_points,1)) * 600 * (2*np.pi)/360
    d1x = -1.5*np.cos(n)*n + np.random.randn(n_points,1) * noise
    d1y =  1.5*np.sin(n)*n + np.random.randn(n_points,1) * noise
    return (np.vstack((np.hstack((d1x,d1y)),np.hstack((-d1x,-d1y)))),
            np.hstack((np.zeros(n_points),np.ones(n_points))))

def makemoons(n_points, noise=.5, random_state=920):
    """
     Returns the make moons dataset.
    """
    return make_moons(n_samples=n_points, noise=noise, random_state=random_state)

In [ ]:
np.random.permutation(10)

X, Y = makemoons(1500, noise=0.2)
perm = np.random.permutation(1500)
X = X[perm,:]
Y = Y[perm]
num_test = 1000

X.shape

In [ ]:
plt.scatter(X[:,0], X[:,1], c=Y)

In [ ]:
fig, ax = plt.subplots(1, 2)
ax[0].scatter(X[:num_test, 0], X[:num_test, 1], c=Y[:num_test])
ax[1].scatter(X[num_test:,0], X[num_test:, 1], c=Y[num_test:], alpha = 0.5)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = "mps" if torch.backends.mps.is_available() else device
print(f"Using device: {device}")

train_x = torch.FloatTensor(X[:num_test])
test_x = torch.FloatTensor(X[num_test:])

train_y = torch.FloatTensor(Y[:num_test]).unsqueeze(-1)
test_y = torch.FloatTensor(Y[num_test:]).unsqueeze(-1)

## Model Definition and Training

In [ ]:
use_cuda = torch.cuda.is_available()
if use_cuda:
    torch.cuda.set_device(0)
    torch.set_default_tensor_type(torch.cuda.FloatTensor)
    torch.set_default_device(device) # or 'cuda'
    train_x, train_y = train_x.cuda(), train_y.cuda()
    test_x, test_y = test_x.cuda(), test_y.cuda()

train_x = train_x.to(device)
train_y = train_y.to(device)
test_x = test_x.to(device)
test_y = test_y.to(device)

In [ ]:
def train_model(model, train_x, train_y):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_func = torch.nn.BCEWithLogitsLoss()

    losses = []
    trainL = -1

    for step in range(2000):
        optimizer.zero_grad()
        outputs = model(train_x)

        loss=loss_func(outputs, train_y)
        trainL = loss.detach().item()
        # if step % 500 is 0:
        #     print("train loss = ", trainL)
        losses.append(trainL)
        loss.backward()
        optimizer.step()
    # print("train loss = ", trainL)
    return losses

In [ ]:
def get_model(hidden_size=20, n_hidden=5):
    in_dim = 2
    out_dim = 1
    model = hess.nets.SimpleNet(
        in_dim,
        out_dim,
        n_hidden=n_hidden,
        hidden_size=hidden_size,
        activation=torch.nn.CELU(alpha=3.0),
        bias=True,
    )
    return model.to(device)

In [ ]:
def get_hessian(model, train_x, train_y):
    n_par = sum(torch.numel(p) for p in model.parameters())
    model_device = next(model.parameters()).device

    hessian = torch.zeros(n_par, n_par)
    for pp in range(n_par):
        base_vec = torch.zeros(n_par, device=model_device).unsqueeze(0)
        base_vec[0, pp] = 1.0

        base_vec = utils.unflatten_like(base_vec, model.parameters())
        utils.eval_hess_vec_prod(
            base_vec,
            list(model.parameters()),
            model,
            criterion=torch.nn.BCEWithLogitsLoss(),
            inputs=train_x,
            targets=train_y,
            use_cuda=(str(model_device).startswith('cuda')),
        )
        if pp == 0:
            output = utils.gradtensor_to_tensor(model, include_bn=True)
            hessian = torch.zeros(output.nelement(), output.nelement())
            hessian[:, pp] = output.cpu()

        hessian[:, pp] = utils.gradtensor_to_tensor(model, include_bn=True).cpu()
        
    return hessian

### Sequential Training

In [ ]:
rep_full_list = []

for rep in tqdm(range(25), desc='Repetitions'):
    eigs_list = []
    losses_list = []
    num_pars = []
    #hessian_list = []
    test_loss_list = []
    
    for i in tqdm(range(1, 15), desc=f'Depth sweep (rep {rep + 1}/25)', leave=False):
    # for i in i_values:
        # print(f'now running model depth: {i}')

        model = get_model(n_hidden = i)
        n_par = sum(p.numel() for p in model.parameters())

        losses = train_model(model, train_x, train_y)

        hessian = get_hessian(model, train_x, train_y).detach()
        
        with torch.no_grad():
            test_loss = torch.nn.BCEWithLogitsLoss()(model(test_x), test_y)
            # print('test loss:', test_loss.item())
        
        losses_list.append(losses)
        eigs_list.append(np.linalg.eig(hessian.cpu().numpy())[0])
        num_pars.append(n_par)
        #hessian_list.append(hessian)
        test_loss_list.append(test_loss.item())
        del hessian, model
        
    rep_full_list.append([losses_list, eigs_list, test_loss_list])

### CUDA Accelerated Training

In [ ]:
# import torch
# import torch.multiprocessing as mp
# from concurrent.futures import ProcessPoolExecutor
# from tqdm.notebook import tqdm 
# from cuda_worker import run_single_experiment 

# try:
#     mp.set_start_method('spawn')
# except RuntimeError:
#     pass 

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# train_x, train_y = train_x.to(device), train_y.to(device)
# test_x, test_y = test_x.to(device), test_y.to(device)

# # Prepare the ordered task list
# tasks = []
# hidden_sizes = [5, 10, 15, 20]
# model_depths = list(range(0, 15))
# reps = 25

# for rep in range(reps):
#     for model_depth in model_depths:
#         # Test fewer parameters to get ML U-shaped curve
#         if model_depth == 0:
#             for hidden_size in hidden_sizes:
#                 tasks.append((rep, hidden_size, model_depth, train_x, train_y, test_x, test_y, device))
        
#         # Test other model configurations on hidden_size = 20
#         else:
#             tasks.append((rep, 20, model_depth, train_x, train_y, test_x, test_y, device))

# max_concurrent_models = 5
# print(f"Dispatching {len(tasks)} experiments")

# results = []
# with ProcessPoolExecutor(max_workers=max_concurrent_models) as executor:
#     # executor.map preserves the strict ordering of tasks
#     map_results = executor.map(run_single_experiment, tasks)
    
#     for result in tqdm(map_results, total=len(tasks), desc="Training Models"):
#         results.append(result)

# rep_full_list = []
# task_idx = 0

# for rep in range(reps):
#     losses_list, eigs_list, test_loss_list, num_pars = [], [], [], []
#     for model_depth in model_depths:
#         if model_depth == 0:
#             for hidden_size in hidden_sizes:
#                 losses, eigs, test_loss_val, n_par = results[task_idx]
                
#                 losses_list.append(losses)
#                 eigs_list.append(eigs)
#                 test_loss_list.append(test_loss_val)
#                 num_pars.append(n_par)
                
#                 task_idx += 1

#         else:
#             losses, eigs, test_loss_val, n_par = results[task_idx]
            
#             losses_list.append(losses)
#             eigs_list.append(eigs)
#             test_loss_list.append(test_loss_val)
#             num_pars.append(n_par)
            
#             task_idx += 1
        
#     rep_full_list.append([losses_list, eigs_list, test_loss_list])

# print("Finished execution")

## Evaluation

In [ ]:
def eff_dim(x, s = 0.1):
    return np.sum(x / (x + s))

ed_list = []
fl_list = []
for losses_list, eigs_list, _ in rep_full_list:
    eff_dim_arr = np.array([eff_dim(ee, s = 50.) for ee in eigs_list])
    final_loss = [l[-1] for l in losses_list]
    
    ed_list.append(eff_dim_arr)
    fl_list.append(final_loss)
    
ed_list = np.array(ed_list)
fl_list = np.array(fl_list)
tl_list = np.array([ee[-1] for ee in rep_full_list])

In [ ]:
plt.plot(fl_list)

In [ ]:
fig, ax = plt.subplots()

# ax.plot(num_pars, ed_list.mean(0), label = 'Effective Dim', color = 'green')
# ax.fill_between(num_pars, ed_list.mean(0) - 2 * ed_list.std(0), ed_list.mean(0) + 2*ed_list.std(0), alpha = 0.3,
#                color='green')
# ax.set_ylabel('Effective Dimensionality', fontsize = 16)
# ax.set_ylim((0, 20))

ax2 = ax.twinx()
ax2.plot(num_pars, fl_list.mean(0), label = 'Final Loss')
ax2.fill_between(num_pars, fl_list.mean(0) - 2 * fl_list.std(0), fl_list.mean(0) + 2*fl_list.std(0), alpha = 0.3)

ax2.plot(num_pars, tl_list.mean(0), label = 'Test Loss')
ax2.fill_between(num_pars, tl_list.mean(0) - 2 * tl_list.std(0), tl_list.mean(0) + 2*tl_list.std(0), alpha = 0.3)
ax2.set_ylabel('Loss', fontsize = 16)
ax2.legend()

ax.set_xlabel('Number of Parameters', fontsize = 16)
fig.suptitle('Increasing Depth', fontsize = 24)
#plt.semilogy()
#plt.ylim(0, 3)
#plt.vlines(train_x.shape[0], 0, 3)

In [ ]:
plt.plot(num_pars, tl_list.mean(0), label = 'Test Loss', color= 'blue')
plt.fill_between(num_pars, tl_list.mean(0) - 2 * tl_list.std(0), tl_list.mean(0) + 2*tl_list.std(0), alpha = 0.3)

plt.plot(num_pars, tl_list.T, color = 'blue', alpha = 0.1)

In [ ]:
plt.plot(num_pars, ed_list.mean(0), label = 'Effective Dim')
plt.fill_between(num_pars, ed_list.mean(0) - 2 * ed_list.std(0), ed_list.mean(0) + 2*ed_list.std(0), alpha = 0.3)

plt.plot(num_pars, ed_list.T, color = 'blue', alpha = 0.02)
plt.ylim((0, 16))

In [ ]:
plt.plot(num_pars, tl_list.T)

## Save/Load Saved Files

In [ ]:
import pickle

with open('./saved_files/moons_depth_celu.pkl', 'wb') as handle:
    pickle.dump([rep_full_list, num_pars], handle, pickle.HIGHEST_PROTOCOL)

In [ ]:
import pickle

with open('./saved_files/moons_depth_celu.pkl', 'rb') as handle:
    rep_full_list, num_pars = pickle.load(handle)

## Decision Boundary Visualization

In [ ]:
# Convert parameters to number of layers
# Formula (for width 20): 60 (input) + 420 * (layers - 1) + 42 (output) = params
# layers = ((params - 102) / 420) + 1
def get_layer_count(p):
    return int(round((p - 102) / 420)) + 1 if p is not None else None

# Model configuration at three critical points of the double descent curve
def extract_critical_configs(rep_full_list, num_pars):
    test_losses = np.mean([rep[2] for rep in rep_full_list], axis=0)
    num_pars = np.array(num_pars)
    
    # Find the indices
    max_loss_idx = np.argmax(test_losses)
    pars_at_max = num_pars[max_loss_idx]
    
    if max_loss_idx > 0:
        first_min_idx = np.argmin(test_losses[:max_loss_idx])
        pars_at_first_min = num_pars[first_min_idx]
    else:
        pars_at_first_min = None
        
    if max_loss_idx < len(test_losses) - 1:
        next_min_idx = max_loss_idx + 1 + np.argmin(test_losses[max_loss_idx + 1:])
        pars_at_next_min = num_pars[next_min_idx]
    else:
        pars_at_next_min = None

    depth_at_first_min = get_layer_count(pars_at_first_min)
    depth_at_max = get_layer_count(pars_at_max)
    depth_at_next_min = get_layer_count(pars_at_next_min)
    
    return [depth_at_first_min, depth_at_max, depth_at_next_min]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

# Model configuration
hidden_size = 20
depth_configs = [0, 4, 12]
print(f"Critical depths: {depth_configs}")

# Keep the worst (highest test loss) seen
max_retries = {1: 30}  # index into depth_configs -> num retries

loss_func = torch.nn.BCEWithLogitsLoss()

for i, n_hidden in enumerate(depth_configs):
    n_retries = max_retries.get(i, 1)
    
    best_model = None
    best_test_loss = -1.0

    for attempt in range(n_retries):
        model = get_model(hidden_size=hidden_size, n_hidden=n_hidden).to(device)
        train_model(model, train_x, train_y)

        model.eval()
        with torch.no_grad():
            out_test = model(test_x)
            te_y = test_y.view(-1, 1) if out_test.dim() == 2 and test_y.dim() == 1 else test_y
            loss_test = loss_func(out_test, te_y).item()

        print(f"  depth={n_hidden}, attempt={attempt+1}/{n_retries}, test_loss={loss_test:.4f}")

        if loss_test > best_test_loss:
            best_test_loss = loss_test
            import copy
            best_model = copy.deepcopy(model)

    model = best_model
    print(f"\nUsing model with best (highest) test loss = {best_test_loss:.4f}")

    params = sum(p.numel() for p in model.parameters())
    print(f"Model: n_hidden={n_hidden} (params={params})")

    # Compute train metrics
    model.eval()
    with torch.no_grad():
        out_train = model(train_x)
        t_y = train_y.view(-1, 1) if out_train.dim() == 2 and train_y.dim() == 1 else train_y
        loss_train = loss_func(out_train, t_y).item()
        preds_train = (torch.sigmoid(out_train) > 0.5).float()
        acc_train = (preds_train == t_y).float().mean().item()

        out_test = model(test_x)
        te_y = test_y.view(-1, 1) if out_test.dim() == 2 and test_y.dim() == 1 else test_y
        loss_test = loss_func(out_test, te_y).item()
        preds_test = (torch.sigmoid(out_test) > 0.5).float()
        acc_test = (preds_test == te_y).float().mean().item()

    print(f"Train Loss: {loss_train:.4f}, Train Acc: {acc_train*100:.2f}%")
    print(f"Test Loss:  {loss_test:.4f}, Test Acc:  {acc_test*100:.2f}%")

    # Decision boundary mesh
    h = 0.05
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

    grid_tensor = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()]).to(device)
    with torch.no_grad():
        Z = torch.sigmoid(model(grid_tensor))
        preds = (Z > 0.5).float()

    Z = preds.cpu().numpy().reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, alpha=0.6, cmap=plt.cm.coolwarm)
    plt.scatter(X[:, 0], X[:, 1], c=Y, edgecolors='k', cmap=plt.cm.coolwarm)
    plt.title(f"Decision Boundary (hidden_size={hidden_size}, n_hidden={n_hidden+1})\n"
              f"Test Loss={loss_test:.4f}, Test Acc={acc_test*100:.1f}%")
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.show()